# **1. Mounting Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **2. Install Dependencies for PySpark**

Installed Java, PySpark, and findspark to set up Spark in this new Colab session. This is required every time a new runtime starts.

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# **3. SparkSession Initialisation**

Initialised the SparkSession with 8GB of memory and 200 shuffle partitions, matching the configuration used in the preprocessing notebook. This keeps the resource discussion consistent with what actually processed this data.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("AmazonAutomotiveOptimisation")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

Spark version : 4.0.3
Spark UI      : http://3dbbae769fdb:4040


# **4. Loading the Preprocessed Parquet Data**

I loaded the training and test Parquet files to confirm the data is accessible before running any Spark UI or caching demonstrations.

In [ ]:
output_dir = "/content/drive/MyDrive/amazon_automotive_sentiment"

training_data = spark.read.parquet(f"{output_dir}/training_data.parquet")
testing_data  = spark.read.parquet(f"{output_dir}/testing_data.parquet")

print(f"Training rows : {training_data.count():,}")
print(f"Testing rows  : {testing_data.count():,}")

Training rows : 15,763,810
Testing rows  : 3,943,422


# **5. Resource Configuration Justification**

I printed the active Spark configuration values to confirm what was actually running, then explained the reasoning behind each one. The 8GB driver memory accounts for the dataset's size and the 65,541 dimension sparse vectors, while 200 shuffle partitions was chosen to spread that high-dimensional data evenly across tasks during processing.

In [ ]:
print("Default parallelism :", spark.sparkContext.defaultParallelism)

for key in ["spark.driver.memory", "spark.executor.memory",
            "spark.sql.shuffle.partitions", "spark.master"]:
    print(f"{key:32s} = {spark.conf.get(key, '(unset)')}")

print()
print("driver.memory = 8g")
print("  The Automotive dataset is 8.3 GB uncompressed, with 65,541")
print("  dimension sparse TF-IDF vectors across 15.7M training rows.")
print()
print("shuffle.partitions = 200")
print("  Chosen to evenly distribute the high-dimensional sparse vectors")
print("  across partitions during pipeline transformations.")
print()
print("master = local[*]")
print("  Uses all available CPU cores for distributed tokenisation,")
print("  TF-IDF fitting, and model training.")

Default parallelism : 2
spark.driver.memory              = 8g
spark.executor.memory            = (unset)
spark.sql.shuffle.partitions     = 200
spark.master                     = local[*]

driver.memory = 8g
  The Automotive dataset is 8.3 GB uncompressed, with 65,541
  dimension sparse TF-IDF vectors across 15.7M training rows.

shuffle.partitions = 200
  Chosen to evenly distribute the high-dimensional sparse vectors
  across partitions during pipeline transformations.

master = local[*]
  Uses all available CPU cores for distributed tokenisation,
  TF-IDF fitting, and model training.


# **6. Opening the Spark UI**

I used ngrok to expose Spark's UI on port 4040, since Colab doesn't allow direct access to localhost ports. I screenshotted the Jobs, Stages, and Storage tabs from this link as evidence.

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok

ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")
public_url = ngrok.connect(4040)
print(f"Spark UI: {public_url}")

Spark UI: NgrokTunnel: "https://reassure-catapult-compactly.ngrok-free.dev" -> "http://localhost:4040"


# **7. Repartitioning Strategy**

I compared repartition, which performs a full shuffle, against coalesce and merges partitions without one. This shows how partition count changes depending on which method is used.

In [ ]:
cores = spark.sparkContext.defaultParallelism

training_repartitioned = training_data.repartition(cores * 2)
training_coalesced = training_repartitioned.coalesce(cores)

print(f"Cores              : {cores}")
print(f"After repartition  : {training_repartitioned.rdd.getNumPartitions()}")
print(f"After coalesce     : {training_coalesced.rdd.getNumPartitions()}")

Cores              : 2
After repartition  : 4
After coalesce     : 2


# **8. Caching Baseline — No Cache**

I measured two identical groupBy operations on the uncached training data, to establish a baseline before comparing against cached performance.

In [ ]:
import time

start_time = time.time()
training_data.groupBy("sentiment").count().collect()
no_cache_run1 = time.time() - start_time

start_time = time.time()
training_data.groupBy("sentiment").count().collect()
no_cache_run2 = time.time() - start_time

print(f"No-cache run1 : {no_cache_run1:.2f}s")
print(f"No-cache run2 : {no_cache_run2:.2f}s")

No-cache run1 : 14.71s
No-cache run2 : 6.34s


# **9. Persisting with MEMORY_AND_DISK**

I persisted the training data with MEMORY_AND_DISK so any portion that doesn't fit in memory spills to disk rather than being recomputed. I then repeated the same groupBy operation to compare timing against the uncached baseline.

In [ ]:
from pyspark import StorageLevel

training_data.persist(StorageLevel.MEMORY_AND_DISK)
training_data.count()

print(f"Storage level : {training_data.storageLevel}")

start_time = time.time()
training_data.groupBy("sentiment").count().collect()
cached_run1 = time.time() - start_time

start_time = time.time()
training_data.groupBy("sentiment").count().collect()
cached_run2 = time.time() - start_time

print(f"Cached run1 : {cached_run1:.2f}s")
print(f"Cached run2 : {cached_run2:.2f}s")

Storage level : Disk Memory Serialized 1x Replicated
Cached run1 : 8.09s
Cached run2 : 9.36s


# **10. Verifying the Cache via Physical Plan**

I checked the physical execution plan to confirm Spark is reading from the in-memory cache rather than re-scanning the Parquet file from disk.

In [ ]:
training_data.groupBy("sentiment").count().explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[sentiment#0], functions=[count(1)])
   +- Exchange hashpartitioning(sentiment#0, 200), ENSURE_REQUIREMENTS, [plan_id=379]
      +- HashAggregate(keys=[knownfloatingpointnormalized(normalizenanandzero(sentiment#0)) AS sentiment#0], functions=[partial_count(1)])
         +- InMemoryTableScan [sentiment#0]
               +- InMemoryRelation [sentiment#0, features#1], StorageLevel(disk, memory, 1 replicas)
                     +- *(1) ColumnarToRow
                        +- FileScan parquet [sentiment#0,features#1] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/7006SCN_Automotive/training_data.parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sentiment:double,features:struct<type:tinyint,size:int,indices:array<int>,values:array<dou...




# **11. Releasing the Cache**

In [ ]:
training_data.unpersist()
print(f"Storage level after unpersist : {training_data.storageLevel}")

Storage level after unpersist : Serialized 1x Replicated


# **12. Spark Session Termination**

In [ ]:
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.
